In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from blox2.utils import make_scaled_trajectory

plt.rcParams.update({
    "font.size": 16,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16,
})

init_path = "results/initial_observed_properties.csv"
all_path = "../data/zinc_dft/properties.csv"
n_obs_0 = 10

dir_names = [
    "random",
    "rf_bpred_nes10_s1",
    "lgb_def_nes100_s1",
    "oracle_s1",
    "oracle_truescale_s1"
]

trajectories = []
for dn in dir_names:
    t = make_scaled_trajectory(init_path, f"results/{dn}/observation_history.csv", all_path)
    trajectories.append(t)
    plt.scatter(t[:, 0], t[:, 1], label=dn, alpha=0.5, s=1)

plt.legend()

In [ ]:
# convex hull area
from blox2 import convex_hull_area_trajectory

for i, t in enumerate(trajectories):
    ch_trj = convex_hull_area_trajectory(t)
    plt.plot(ch_trj[n_obs_0:], label=dir_names[i])

plt.xscale("log", base=10)
# plt.yscale("log", base=10)
plt.rcParams.update({"legend.fontsize": 12})
plt.legend()

In [ ]:
# convex hull perimeter
from blox2 import convex_hull_perimeter_trajectory

for i, t in enumerate(trajectories):
    ch_trj = convex_hull_perimeter_trajectory(t)
    plt.plot(ch_trj[n_obs_0:], label=dir_names[i])

plt.xscale("log", base=10)
# plt.yscale("log", base=10)
plt.rcParams.update({"legend.fontsize": 12})
plt.legend()

In [ ]:
def occupancy_prefix(X: np.ndarray, X_all: np.ndarray=None, bins: int=50, bounds: tuple[tuple[float, float], tuple[float, float]]=None, pad: float=0.0) -> np.ndarray:
    """
    Grid occupancy ratio for each prefix X[:k] (k=1..N) in 2D.

    Occupancy is (# unique grid cells hit by points) / (total grid cells),
    where the grid bounds are fixed by:
      - bounds if provided, else
      - X_all if provided, else
      - X (i.e., only observed/prefix data; not recommended for stable trajectory)

    Args:
        X : (N, 2) points in time/order.
        X_all : (M, 2) points used to fix the grid bounds (recommended).
        bins : number of bins per axis (bins x bins grid).
        bounds : ((xmin,xmax),(ymin,ymax)) explicit bounds (overrides X_all).
        pad : extend bounds by pad * range (e.g., 0.05 adds 5% margin each side).

    Returns:
        occ: (N,) occupancy trajectory in [0,1]
    """
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or X.shape[1] != 2:
        raise ValueError(f"X must be shape (N,2), got {X.shape}")
    if X_all is None:
        X_ref = X
    else:
        X_ref = np.asarray(X_all, dtype=float)
        if X_ref.ndim != 2 or X_ref.shape[1] != 2:
            raise ValueError(f"X_all must be shape (M,2), got {X_ref.shape}")

    if bounds is None:
        xmin, ymin = np.min(X_ref, axis=0)
        xmax, ymax = np.max(X_ref, axis=0)
    else:
        (xmin, xmax), (ymin, ymax) = bounds

    # padding
    xr = xmax - xmin
    yr = ymax - ymin
    xmin = xmin - pad * xr
    xmax = xmax + pad * xr
    ymin = ymin - pad * yr
    ymax = ymax + pad * yr

    # handle degenerate range
    eps = 1e-12
    xr = max(xmax - xmin, eps)
    yr = max(ymax - ymin, eps)

    N = X.shape[0]
    occ = np.zeros(N, dtype=float)

    # map points -> integer cell indices (0..bins-1)
    # stable even if points fall outside bounds (clip)
    def to_cell_ids(pts: np.ndarray) -> np.ndarray:
        u = (pts[:, 0] - xmin) / xr
        v = (pts[:, 1] - ymin) / yr
        ix = np.clip((u * bins).astype(int), 0, bins - 1)
        iy = np.clip((v * bins).astype(int), 0, bins - 1)
        # combine into single id
        return ix * bins + iy

    total_cells = bins * bins

    for k in range(1, N + 1):
        ids = to_cell_ids(X[:k])
        occ[k - 1] = np.unique(ids).size / total_cells

    return occ